# Planner LLM 테스트 노트북

이 노트북은 `Planner/planner_service.py`를 직접 호출해 Planner 단계만 실험합니다.
입력은 질문 1개만 사용하며, DB 실행/SQL 생성/RAG 검색 실행은 포함하지 않습니다.

## 1) 프로젝트 루트 경로 설정

실행 위치가 달라도 `Planner`, `Root_Stream` 모듈을 import할 수 있도록 루트 경로를 찾습니다.

In [3]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
project_root = None
for candidate in [current, *current.parents]:
    if (candidate / "Planner").exists() and (candidate / "Root_Stream").exists():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError("프로젝트 루트를 찾을 수 없습니다. DB_TO_LLM 루트에서 다시 실행하세요.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project_root={project_root}")

project_root=C:\Users\김민한\Desktop\개발\DB_to_LLM


## 2) import 및 환경 설정

Planner 서비스와 검증 함수를 import하고 config 경로를 준비합니다.

In [4]:
import json

from Planner.planner_service import PlannerService
from Planner.plan_validator import validate_plan_payload

config_path = project_root / "Root_Stream" / "config" / "config.yaml"
planner_service = PlannerService(config_path=config_path)
print(f"config_path={config_path}")

[2026-04-10 09:50:57] [INFO] [config_loader.py:load_config:21] 설정 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-10 09:50:57] [INFO] [config_loader.py:load_config:30] 설정 로드 완료
[2026-04-10 09:50:57] [INFO] [prompt_manager.py:_load_prompts:34] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_templates.yaml
[2026-04-10 09:50:57] [INFO] [prompt_manager.py:_load_prompts:46] 프롬프트 파일 로드 완료: prompt_count=5
[2026-04-10 09:50:57] [INFO] [planner_service.py:__init__:44] PlannerService 초기화 완료: config=C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml, llm_provider=ollama


config_path=C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml


## 3) 단일 질문 테스트

질문 1개를 입력해 raw response와 parsed plan을 확인합니다.

In [6]:
question = "최근 30일간 가장 많이 발생한 알람을 찾고 해당 알람 정보를 알려줘"
result = planner_service.plan_question(question)

print("[QUESTION]")
print(result.question)
print("\n[RAW RESPONSE]")
print(result.raw_response)
print("\n[PARSED PLAN]")
print(json.dumps(result.plan.to_dict(), ensure_ascii=False, indent=2))

[2026-04-10 09:51:30] [INFO] [planner_service.py:plan_question:71] Planner 실행 시작: question_length=38
[2026-04-10 09:51:30] [INFO] [ollama_client.py:generate:122] Ollama 호출 시작: model=qwen2.5:14b, endpoint=http://192.168.0.74:11434/api/chat
[2026-04-10 09:51:37] [INFO] [ollama_client.py:generate:170] Ollama 호출 완료: output_length=355
[2026-04-10 09:51:37] [INFO] [planner_service.py:plan_question:89] Planner 실행 완료: query_type=DB_THEN_RAG, step_count=2


[QUESTION]
최근 30일간 가장 많이 발생한 알람을 찾고 해당 알람 정보를 알려줘

[RAW RESPONSE]
{
  "is_composite": true,
  "query_type": "DB_THEN_RAG",
  "steps": [
    {
      "step": 1,
      "type": "db",
      "goal": "최근 30일간 발생한 알람 건수를 집계하여 가장 많이 발생한 알람 타입을 찾음",
      "depends_on": []
    },
    {
      "step": 2,
      "type": "rag",
      "goal": "가장 많이 발생한 알람의 상세 정보를 제공하는 문서나 가이드를 찾아냄",
      "depends_on": [
        1
      ]
    }
  ]
}

[PARSED PLAN]
{
  "is_composite": true,
  "query_type": "DB_THEN_RAG",
  "steps": [
    {
      "step": 1,
      "type": "db",
      "goal": "최근 30일간 발생한 알람 건수를 집계하여 가장 많이 발생한 알람 타입을 찾음",
      "depends_on": []
    },
    {
      "step": 2,
      "type": "rag",
      "goal": "가장 많이 발생한 알람의 상세 정보를 제공하는 문서나 가이드를 찾아냄",
      "depends_on": [
        1
      ]
    }
  ]
}


## 4) validation 결과 확인

파싱된 plan을 validator에 넣어 최소 형식 검증 결과를 확인합니다.

In [ ]:
plan_payload = result.plan.to_dict()
validate_plan_payload(plan_payload)
print("validation=OK")
print(json.dumps(plan_payload, ensure_ascii=False, indent=2))